# 03 · Definitieve datavoorbereiding

**Auteurs:** _vul namen en concrete bijdragen in vóór indiening_  
**Doel:** Zonder grafieken de volledige raw→prepared-transformatie reproduceerbaar uitvoeren.

Elke codecel wordt voorafgegaan door uitleg. Voer de notebooks in nummervolgorde uit
vanaf de repository-root. Resultaten moeten door het team zelf worden gecontroleerd en
geïnterpreteerd; synthetische CI-data gelden nooit als onderzoeksbewijs.

## Eén bron van waarheid

Alle cleaningregels leven in `src/data/prepare.py`; dit notebook roept die code aan en voert acceptatiechecks uit.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from src.config import RAW_DIR, PROCESSED_DIR
from src.data.prepare import prepare_archives

OUTPUT = PROCESSED_DIR / "trips.parquet"
summary = prepare_archives(RAW_DIR, OUTPUT)
display(pd.Series({k: v for k, v in summary.items() if k != "chunks"}))

## Acceptatiechecks

Deze invarianten voorkomen dat schemawijzigingen stilletjes verkeerde trainingsdata opleveren.

In [ ]:
data = pd.read_parquet(OUTPUT)
assert len(data) == summary["output_rows"]
assert data.ride_id.is_unique
assert data.duration_minutes.between(1, 180).all()
assert data.member_casual.isin(["member", "casual"]).all()
assert data.start_lat.between(40.4, 41.1).all()
assert data.start_lng.between(-74.4, -73.5).all()
assert data.started_at.notna().all()
assert data.start_station_id.notna().all()
print(f"PASS — {len(data):,} rijen, {data.started_at.min()} t/m {data.started_at.max()}")

## Outputcontract

De Parquet-data en audit worden niet gecommit vanwege omvang. Het downloadmanifest, codeversie en requirements maken reconstructie mogelijk.